# PortBlend Python SDK Quickstart Cookbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rulesense/portblend/blob/main/doc/examples/01_quickstart_portblend.ipynb)

Official interactive tutorial for **PortBlend** — strategy pairwise correlation calculation, SLSQP portfolio weight optimization, and drawdown minimization.

In [ ]:
# 1. Install PortBlend SDK and plotting dependencies
!pip install portblend pandas matplotlib seaborn --quiet

## 2. Load Sample Multi-Strategy NAV Dataset

Generate synthetic daily NAV series for 3 trading strategies: Trend Following, Mean Reversion, and Commodity Momentum.

In [ ]:
# Initialize Client (Defaults to production https://app.portblend.com/api)
client = PortBlendClient(api_key="pb_live_demo")
# For local development server testing:
# client = PortBlendClient(api_key="pb_live_demo", base_url="http://localhost:8000/api")

# Compute Correlation Matrix
df_corr = client.correlate(data=df_strategies)

print("\nPairwise Correlation Matrix:")
print(df_corr)

# Heatmap Plot
plt.figure(figsize=(7, 5))
sns.heatmap(df_corr, annot=True, cmap="coolwarm", vmin=-1.0, vmax=1.0, fmt=".2f", linewidths=1)
plt.title("PortBlend Strategy Pairwise Correlation Matrix", fontsize=12, fontweight="bold")
plt.show()

## 3. Compute Pairwise Strategy Correlation Matrix

Pass the strategy DataFrame to `client.correlate()` to compute the pairwise Pearson correlation matrix across returns.

In [ ]:
# Initialize Client (Pass your developer API Key 'pb_live_...')
client = PortBlendClient(api_key='pb_live_demo', base_url='http://localhost:8000/api')

# Compute Correlation Matrix
df_corr = client.correlate(data=df_strategies)

print('\nPairwise Correlation Matrix:')
print(df_corr)

# Heatmap Plot
plt.figure(figsize=(7, 5))
sns.heatmap(df_corr, annot=True, cmap='coolwarm', vmin=-1.0, vmax=1.0, fmt='.2f', linewidths=1)
plt.title('PortBlend Strategy Pairwise Correlation Matrix', fontsize=12, fontweight='bold')
plt.show()

## 4. Optimize Portfolio Weights for Minimum Drawdown

Run `client.blend()` with `target="min_drawdown"` and `allow_cash=True`.

In [ ]:
# Execute SLSQP Weight Optimization
result = client.blend(
    data=df_strategies,
    target='min_drawdown',  # Options: 'min_drawdown' | 'max_sharpe' | 'max_calmar' | 'min_volatility' | 'max_sortino'
    allow_cash=True
)

# Educational Quantitative Insight Synthesis
result.summary()

# Access Optimal Weights Dictionary
optimal_weights = result.weights
print('\nOptimal Allocation Weights (%):')
for strat, w in optimal_weights.items():
    print(f'  - {strat}: {w}%')

## 5. Plot Blended Equity Curve and Performance Comparison

In [ ]:
# Calculate Blended Portfolio NAV Series
w_a = optimal_weights.get('Trend_Follower', 0.0) / 100.0
w_b = optimal_weights.get('Mean_Reversion', 0.0) / 100.0
w_c = optimal_weights.get('Commodity_Momentum', 0.0) / 100.0

rets_a = df_strategies['Trend_Follower'].pct_change().fillna(0)
rets_b = df_strategies['Mean_Reversion'].pct_change().fillna(0)
rets_c = df_strategies['Commodity_Momentum'].pct_change().fillna(0)

port_rets = w_a * rets_a + w_b * rets_b + w_c * rets_c
blended_nav = 100.0 * np.cumprod(1.0 + port_rets)

# Plot Equity Curves
plt.figure(figsize=(12, 6))
plt.plot(dates, df_strategies['Trend_Follower'], label='Trend Follower', alpha=0.5, linestyle='--')
plt.plot(dates, df_strategies['Mean_Reversion'], label='Mean Reversion', alpha=0.5, linestyle='--')
plt.plot(dates, df_strategies['Commodity_Momentum'], label='Commodity Momentum', alpha=0.5, linestyle='--')
plt.plot(dates, blended_nav, label='PortBlend Blended Portfolio', color='#4f46e5', linewidth=2.5)

plt.title('PortBlend Strategy Blending — Equity Curve Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Normalized NAV (Base 100)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()